# Evaluación 3 · Sección 6 — Base de análisis

Continuación de `Contreras_Daniela-Nicklichen_Emanuel_Evaluacion_2_S5_Hallazgos.ipynb`.

**Alcance de este notebook:** construir una única tabla con una fila por cada postulante que no fue seleccionado en ninguna preferencia, con todas las variables que los análisis y el modelo de la Sección 7 necesitan. No produce hallazgos: produce el insumo.

## 6.1 Por qué existe este notebook

En la Evaluación 2, el cálculo de "alternativa alcanzable" se reconstruyó dos veces: una en la Sección 3 y otra en la Sección 4. Funcionó, pero cada reconstrucción es una oportunidad de que dos notebooks reporten cifras distintas para lo mismo.

Acá el cálculo se hace **una sola vez** y se guarda en `data/processed/base_no_seleccionados.csv`. La Sección 7 lee ese archivo y no vuelve a calcular nada.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

DIRECTORIO_ACTUAL = Path.cwd()
DIRECTORIO_RAIZ = DIRECTORIO_ACTUAL.parent
RAW_DIR = DIRECTORIO_RAIZ / "data" / "raw" / "2026"

carpetas = {
    "Inscritos":       "PAES-2026-Inscritos-Puntajes",
    "Socioeconomico":  "PAES-2026-Socioeconomicos",
    "Postulantes":      "PAES-2026-Postulantes",
    "Matricula":        "PAES-2026-Matricula",
    "Oferta":           "PAES-2026-Oferta-Definitiva-Programas",
}

# Se utiliza glob() en vez de nombre de archivo fijo,
# para evitar errores del tipo FileNotFoundError.
bases = {}
for nombre, carpeta in carpetas.items():
    archivo = list((RAW_DIR / carpeta).glob("*.csv"))[0]
    if nombre == "Oferta":
        bases[nombre] = pd.read_csv(archivo, sep=",", decimal=".")
    else:
        bases[nombre] = pd.read_csv(archivo, sep=";", decimal=",", low_memory=False)

A, B, C, D, O = bases["Inscritos"], bases["Socioeconomico"], bases["Postulantes"], bases["Matricula"], bases["Oferta"]

# Mapeo de codigo a nombre de dependencia, el mismo de las Secciones 2 y 3
dep_map = {"1":"Corp. Municipal","2":"Municipal","3":"Part. Subvencionado",
           "4":"Part. Pagado","5":"Corp. Adm. Delegada","6":"SLEP"}
A["DEP_NOMBRE"] = A["DEPENDENCIA"].astype(str).str.strip().map(dep_map)

print("Bases cargadas:", {k: v.shape for k, v in bases.items()})

## 6.2 La población: postulantes sin selección

Un estudiante quedó fuera si ninguna de sus preferencias terminó en los estados `24` (seleccionado) o `26` (seleccionado en una preferencia anterior). El criterio es el mismo de la Sección 3.

In [ ]:
quedo_seleccionado = C.groupby("MRUN")["ESTADO_PREF"].apply(lambda x: x.isin([24, 26]).any())
mrun_sin_seleccion = quedo_seleccionado[~quedo_seleccionado].index

print("Postulantes totales:", C["MRUN"].nunique())
print("Sin seleccion en ninguna preferencia:", len(mrun_sin_seleccion))

## 6.3 Programas candidatos y oportunidades

El área de interés de cada estudiante son los nombres de carrera que él mismo incluyó en su lista, en cualquier universidad. Un programa es candidato si ofrece una de esas carreras, tiene matrícula final por debajo de sus vacantes informadas, y tiene un corte regular calculable.

In [ ]:
# Area de interes: nombres de CARRERA distintos en la propia lista de cada estudiante
C_con_carrera = C.merge(O[["COD_CARRERA","CARRERA"]],
                        left_on="COD_CARRERA_PREF", right_on="COD_CARRERA", how="left")
interes = C_con_carrera[C_con_carrera["MRUN"].isin(mrun_sin_seleccion)][["MRUN","CARRERA"]].drop_duplicates()

# Corte real de cada programa: el menor puntaje entre sus seleccionados por via REGULAR
seleccionados = C[(C["TIPO_PREF"] == "REGULAR") & (C["ESTADO_PREF"] == 24)].copy()
seleccionados["ptje_real"] = pd.to_numeric(
    seleccionados["PTJE_PREF"].str.replace(",", ".", regex=False), errors="coerce")
corte = seleccionados.groupby("COD_CARRERA_PREF")["ptje_real"].min().rename("corte_real")

# Cupos sin llenar: vacantes informadas menos matricula final observada
matriculados = D.groupby("CODIGO_CARRERA").size().rename("matriculados")
oferta = O.merge(matriculados, left_on="COD_CARRERA", right_index=True, how="left")
oferta = oferta.merge(corte, left_on="COD_CARRERA", right_index=True, how="left")
oferta["matriculados"] = oferta["matriculados"].fillna(0)
oferta["cupos_sin_llenar"] = (oferta["VAC_1ER"] - oferta["matriculados"]).clip(lower=0)

candidatos = oferta[(oferta["cupos_sin_llenar"] > 0) & (oferta["corte_real"].notna())]

print("Programas candidatos:", len(candidatos), "de", len(O))
print("Pares (estudiante, carrera de interes):", len(interes))

## 6.4 Puntaje ponderado de cada oportunidad

Se usa la misma fórmula validada en la Sección 3, que reprodujo el 99,96% de 1.328.381 puntajes oficiales con error menor a 0,5 puntos. Los dos ajustes ya documentados se mantienen: Historia y Ciencias como cupo único, y redistribución de ponderaciones para quienes no tienen NEM chileno.

In [ ]:
oportunidades = interes.merge(candidatos, on="CARRERA", how="inner")
oportunidades = oportunidades.merge(
    A[["MRUN","PTJE_NEM","PTJE_RANKING","CLEC_MAX","MATE1_MAX","MATE2_MAX","HCSOC_MAX","CIEN_MAX"]],
    on="MRUN", how="left")

# Historia/Ciencias: cupo unico. Si el programa pide ambas, se usa el mejor puntaje.
w_hsco = oportunidades["HSCO"].fillna(0)
w_cien = oportunidades["CIEN"].fillna(0)
score_hc = np.select(
    [(w_hsco > 0) & (w_cien > 0), (w_hsco > 0) & (w_cien == 0), (w_cien > 0) & (w_hsco == 0)],
    [oportunidades[["HCSOC_MAX","CIEN_MAX"]].max(axis=1),
     oportunidades["HCSOC_MAX"], oportunidades["CIEN_MAX"]],
    default=0)
peso_hc = np.maximum(w_hsco, w_cien)

# Extranjeros sin NEM: se redistribuye el peso de NEM+Ranking entre las demas pruebas
es_extranjero_sin_nem = (oportunidades["PTJE_NEM"] == 0) | (oportunidades["PTJE_RANKING"] == 0)
peso_clec = oportunidades["CLEC"].fillna(0)
peso_m1 = oportunidades["M1"].fillna(0)
peso_m2 = oportunidades["M2"].fillna(0)
resto = peso_clec + peso_m1 + peso_hc + peso_m2
factor = np.where(es_extranjero_sin_nem & (resto > 0), 100 / resto, 1)

numerador = (
    oportunidades["PTJE_NEM"]     * np.where(es_extranjero_sin_nem, 0, oportunidades["NEM"].fillna(0)) +
    oportunidades["PTJE_RANKING"] * np.where(es_extranjero_sin_nem, 0, oportunidades["RANKING"].fillna(0)) +
    oportunidades["CLEC_MAX"]     * np.where(es_extranjero_sin_nem, peso_clec * factor, peso_clec) +
    oportunidades["MATE1_MAX"]    * np.where(es_extranjero_sin_nem, peso_m1 * factor, peso_m1) +
    score_hc                      * np.where(es_extranjero_sin_nem, peso_hc * factor, peso_hc) +
    oportunidades["MATE2_MAX"]    * np.where(es_extranjero_sin_nem, peso_m2 * factor, peso_m2)
)

oportunidades["puntaje_calculado"] = numerador / 100
oportunidades["margen"] = oportunidades["puntaje_calculado"] - oportunidades["corte_real"]
oportunidades["supera"] = oportunidades["puntaje_calculado"] > oportunidades["corte_real"]

# Version con empate incluido, para el analisis de sensibilidad de la Seccion 7
oportunidades["supera_con_empate"] = oportunidades["puntaje_calculado"] >= oportunidades["corte_real"]

print("Pares (estudiante, oportunidad):", len(oportunidades))
print("Estudiantes con al menos una oportunidad:", oportunidades["MRUN"].nunique())

## 6.5 Resumen por estudiante: tres estados, no dos

En la Evaluación 2 el resultado se reportó como binario. Eso esconde una distinción que sí importa: un estudiante sin ninguna alternativa puede estar en dos situaciones muy distintas — tenía programas candidatos y no alcanzó ninguno, o no tenía ningún programa candidato que evaluar.

In [ ]:
por_estudiante = oportunidades.groupby("MRUN")
resumen_oportunidades = pd.DataFrame({
    "n_oportunidades": por_estudiante.size(),
    "n_supera": por_estudiante["supera"].sum(),
    "n_supera_con_empate": por_estudiante["supera_con_empate"].sum(),
    "mejor_puntaje": por_estudiante["puntaje_calculado"].max(),
    "mejor_margen": por_estudiante["margen"].max(),
})

base = pd.DataFrame({"MRUN": list(mrun_sin_seleccion)})
base = base.merge(resumen_oportunidades, on="MRUN", how="left")

# Quien no aparece en oportunidades no tiene ningun programa candidato: son ceros, no nulos
for col in ["n_oportunidades", "n_supera", "n_supera_con_empate"]:
    base[col] = base[col].fillna(0).astype(int)

base["tiene_alternativa"] = base["n_supera"] > 0
base["estado3"] = np.select(
    [base["n_oportunidades"] == 0, base["n_supera"] == 0],
    ["Sin candidatos evaluables", "Con candidatos, sin alcanzar"],
    default="Con alternativa alcanzable")

tabla_estados = base["estado3"].value_counts().rename("N").to_frame()
tabla_estados["%"] = (tabla_estados["N"] / len(base) * 100).round(2)
tabla_estados

**Decisión documentada:** `mejor_puntaje` queda en nulo para quienes no tienen ningún programa candidato, porque no hay puntaje que calcular — no es un cero. Esto corrige una lectura de la Evaluación 2: el promedio de 713,4 reportado para el grupo "sin alternativa" estaba calculado sobre los 4.405 que sí tenían candidatos, no sobre los 7.156 del grupo completo. Con los tres estados separados, cada cifra queda referida a su población correcta.

## 6.6 Amplitud de la lista de preferencias

Cuántas preferencias usó cada estudiante y cuántas carreras distintas incluyó. Sirven para describir la estrategia de postulación en la Sección 7.

In [ ]:
# n_preferencias cuenta posiciones usadas en la lista; n_carreras_distintas cuenta
# nombres de carrera distintos, que es la amplitud real del area de interes.
# Contar codigos de programa daria casi lo mismo que las posiciones y no informaria nada.
preferencias = C_con_carrera[C_con_carrera["MRUN"].isin(mrun_sin_seleccion)]
amplitud = preferencias.groupby("MRUN").agg(
    n_preferencias=("ORDEN_PREF", "nunique"),
    n_carreras_distintas=("CARRERA", "nunique"),
)
base = base.merge(amplitud, on="MRUN", how="left")

base[["n_preferencias","n_carreras_distintas","n_oportunidades"]].describe().round(1)

## 6.7 Atributos del estudiante

In [ ]:
base = base.merge(A[["MRUN","DEP_NOMBRE","CLEC_MAX","MATE1_MAX"]], on="MRUN", how="left")

# Promedio de las dos pruebas obligatorias: es el criterio que usa el DEMRE para
# la habilitacion, y no depende de que programas eligio el estudiante.
base["ptje_obligatorias"] = (base["CLEC_MAX"] + base["MATE1_MAX"]) / 2

base = base.merge(B[["MRUN","INGRESO_PERCAPITA_GRUPO_FA"]], on="MRUN", how="left")
base = base.rename(columns={"INGRESO_PERCAPITA_GRUPO_FA": "decil"})

# Grupo de la hipotesis, tal como fue escrita en la formulacion
publico = ["Corp. Municipal", "Municipal", "SLEP"]
base["grupo_hipotesis"] = np.select(
    [base["DEP_NOMBRE"] == "Part. Pagado", base["DEP_NOMBRE"].isin(publico)],
    ["Part. Pagado", "Publico"],
    default="Otro")

print(base["grupo_hipotesis"].value_counts(dropna=False))

**Decisión documentada:** `ptje_obligatorias` es el promedio de `CLEC_MAX` y `MATE1_MAX`. Se elige por dos razones: es el criterio que el propio DEMRE usa para determinar la habilitación (documentado en la Sección 2), y no depende de qué programas eligió el estudiante, por lo que no arrastra la construcción del resultado hacia el modelo.

**Decisión documentada:** `grupo_hipotesis` reproduce la hipótesis tal como fue formulada, que compara Particular Pagado contra establecimientos públicos (Corporación Municipal, Municipal y SLEP). Corporación de Administración Delegada y Particular Subvencionado no fueron nombrados en ninguno de los dos lados, así que quedan en `Otro`: aparecen en todos los análisis descriptivos por dependencia, pero no se incorporan al contraste. La hipótesis no se redefine después de ver los resultados.

## 6.8 Guardar y verificar

In [ ]:
PROCESSED_DIR = DIRECTORIO_RAIZ / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
RUTA_BASE = PROCESSED_DIR / "base_no_seleccionados.csv"

base.to_csv(RUTA_BASE, index=False)

print("Guardado en:", RUTA_BASE)
print("Dimensiones:", base.shape)
print()
print("Verificaciones contra la Evaluacion 2:")
print(f"  filas == postulantes sin seleccion : {len(base) == len(mrun_sin_seleccion)}")
print(f"  MRUN unico                         : {base['MRUN'].is_unique}")
print(f"  con alternativa                    : {base['tiene_alternativa'].sum()} ({base['tiene_alternativa'].mean()*100:.2f}%)")
print()
base.dtypes

## 6.9 Qué sigue

La Sección 7 lee `base_no_seleccionados.csv` y ejecuta sobre ella el contraste de la hipótesis, el análisis por decil, la regresión logística y el modelo de mecanismo. Ningún cálculo de esta sección se repite allá.

## Uso de Inteligencia Artificial

Se utilizó **Claude (Anthropic)** como asistente para la construcción de este notebook: la consolidación en una tabla única, la separación del resultado en tres estados y la redacción de las decisiones documentadas. La fórmula del puntaje ponderado y la definición de alternativa alcanzable provienen del trabajo del equipo en la Evaluación 2; la elección de las variables y la definición del grupo de contraste son del equipo. Todas las cifras son reproducibles sobre las bases públicas declaradas.